# 🔄 XGBoost to ONNX Conversion

This notebook converts a trained XGBoost model and StandardScaler to ONNX format for optimized inference.

**Author:** Claude Code  
**Date:** 2025-09-30

---

## Overview

- **Input:** XGBoost model (.joblib), StandardScaler (.joblib), metadata (.json)
- **Output:** ONNX models for scaler and classifier
- **Benefits:** Faster inference, cross-platform deployment, smaller size


## 📦 Imports and Setup

In [ ]:
import json
import warnings
from pathlib import Path
from typing import Dict, Tuple, Optional, Any

import joblib
import numpy as np
import onnx
import onnxruntime as rt
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType as SklearnFloatTensorType

warnings.filterwarnings('ignore')

print("✅ All imports successful")

## ⚙️ Configuration

In [ ]:
# File paths
base_name = "xgboost_compressed_with_features_290925_1025_delivery_dev_run_290925_1025"
onnx_dir = Path(".").resolve()

model_path = onnx_dir / f"{base_name}.joblib"
scaler_path = onnx_dir / f"{base_name}_scaler.joblib"
metadata_path = onnx_dir / f"{base_name}_metadata.json"
output_dir = onnx_dir / "onnx_models"

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Working directory: {onnx_dir}")
print(f"📁 Output directory: {output_dir}")
print(f"\n📄 Input files:")
print(f"  • Model: {model_path.name}")
print(f"  • Scaler: {scaler_path.name}")
print(f"  • Metadata: {metadata_path.name}")

## 📥 Load Original Models

In [ ]:
print("📦 Loading model artifacts...\n")

# Load XGBoost model
if not model_path.exists():
    raise FileNotFoundError(f"Model not found: {model_path}")
model = joblib.load(model_path)
print(f"✓ Model loaded: {type(model).__name__}")

# Load StandardScaler
if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler not found: {scaler_path}")
scaler = joblib.load(scaler_path)
print(f"✓ Scaler loaded: {type(scaler).__name__}")
print(f"✓ Number of features: {scaler.n_features_in_}")

# Load metadata
if not metadata_path.exists():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")
with open(metadata_path, 'r') as f:
    metadata = json.load(f)
print(f"✓ Metadata loaded: {metadata['model_name']}")

n_features = scaler.n_features_in_

print(f"\n📊 Model Information:")
print(f"  • Features: {n_features}")
print(f"  • Test Accuracy: {metadata['test_metrics']['accuracy']:.4f}")
print(f"  • ROC-AUC: {metadata['test_metrics']['roc_auc']:.4f}")

## 🔄 Convert StandardScaler to ONNX

In [ ]:
print("🔄 Converting StandardScaler to ONNX...\n")

scaler_onnx_path = output_dir / "scaler.onnx"

# Define input types
initial_types = [
    ('float_input', SklearnFloatTensorType([None, n_features]))
]

# Convert
onnx_scaler = convert_sklearn(
    scaler,
    initial_types=initial_types,
    target_opset=12
)

# Save
with open(scaler_onnx_path, "wb") as f:
    f.write(onnx_scaler.SerializeToString())

print(f"✓ Scaler converted and saved to: {scaler_onnx_path.name}")

# Validate
onnx_model = onnx.load(scaler_onnx_path)
onnx.checker.check_model(onnx_model)
print(f"✓ ONNX model validation passed")

# Check size
scaler_size = scaler_onnx_path.stat().st_size / 1024
print(f"✓ Model size: {scaler_size:.2f} KB")

## 🔄 Convert XGBoost Model to ONNX

In [ ]:
print("🔄 Converting XGBoost model to ONNX...\n")

model_onnx_path = output_dir / "model.onnx"

# Define input types
initial_types = [
    ('float_input', FloatTensorType([None, n_features]))
]

# Convert using onnxmltools (preferred for XGBoost)
onnx_model = convert_xgboost(
    model,
    initial_types=initial_types,
    target_opset=12
)

# Save
with open(model_onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"✓ Model converted and saved to: {model_onnx_path.name}")

# Validate
onnx_model_check = onnx.load(model_onnx_path)
onnx.checker.check_model(onnx_model_check)
print(f"✓ ONNX model validation passed")

# Check size
model_size = model_onnx_path.stat().st_size / (1024 * 1024)
print(f"✓ Model size: {model_size:.2f} MB")

## 📝 Save ONNX Metadata

In [ ]:
print("📝 Saving ONNX metadata...\n")

metadata_onnx_path = output_dir / "onnx_metadata.json"

onnx_metadata = {
    'original_model': metadata['model_name'],
    'conversion_date': '2025-09-30',
    'n_features': n_features,
    'feature_names': metadata['feature_names'],
    'model_metrics': metadata['test_metrics'],
    'best_params': metadata['best_params'],
    'opset_version': 12,
    'onnx_runtime_version': rt.__version__,
    'input_shape': [None, n_features],
    'input_dtype': 'float32',
    'output_labels': [0, 1],  # Binary classification: Ocean (0) vs Land (1)
    'notes': 'Use scaler_onnx first, then model_onnx for predictions'
}

with open(metadata_onnx_path, 'w') as f:
    json.dump(onnx_metadata, f, indent=2)

print(f"✓ Metadata saved to: {metadata_onnx_path.name}")

## ✅ Conversion Summary

In [ ]:
print("\n" + "="*70)
print("✅ CONVERSION COMPLETE")
print("="*70)
print(f"\nOutput directory: {output_dir}\n")
print(f"Generated files:")
print(f"  • {scaler_onnx_path.name} ({scaler_size:.2f} KB)")
print(f"  • {model_onnx_path.name} ({model_size:.2f} MB)")
print(f"  • {metadata_onnx_path.name}")
print(f"\n💡 Next step: Run notebook 02_test_onnx_models.ipynb to validate conversion")